In [2]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

doc_folder = '../data/documents'
all_docs = []

for filename in os.listdir(doc_folder):
    if filename.endswith('.txt'):
        loader = TextLoader(os.path.join(doc_folder, filename), encoding='utf-8')
        all_docs.extend(loader.load())

print(f"Loaded {len(all_docs)} documents")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(all_docs)

print(f"Split into {len(chunks)} chunks")

Loaded 5 documents
Split into 33 chunks


In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='../data/chroma_db'
)

print("Vector store created successfully")
print("Number of chunks stored:", vectorstore._collection.count())

C:\Users\Asthi\AppData\Local\Temp\ipykernel_23176\410763961.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store created successfully
Number of chunks stored: 33


In [5]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.2")
response = llm.invoke("What is coral bleaching in one sentence?")
print(response)

Coral bleaching is a stress response caused by exposure to high water temperatures, which causes corals to expel the algae that live inside them, turning white and often leading to coral death or decline.


In [7]:
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

prompt_template = """Use the following context about coral reefs to answer the question. If the answer isn't in the context, say you don't have enough information rather than guessing.

Context: {context}

Question: {question}

Answer:"""

PROMPT = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True
)

result = qa_chain.invoke({"query": "What causes coral bleaching?"})
print(result['result'])
print()
print("--- Sources used ---")
for doc in result['source_documents']:
    print(doc.metadata['source'], "-", doc.page_content[:100], "...")

Overexposure to Sunlight combined with high temperatures, intense UV radiation.

--- Sources used ---
../data/documents\coral_bleaching_causes.txt - Coral Bleaching: Causes and Mechanisms ...
../data/documents\coral_bleaching_causes.txt - Coral bleaching occurs when corals expel the symbiotic algae (zooxanthellae) living in their tissues ...
../data/documents\coral_bleaching_causes.txt - 5. Overexposure to Sunlight: Combined with high temperatures, intense UV radiation can compound stre ...


In [8]:
def ask_question(question):
    result = qa_chain.invoke({"query": question})
    print("Q:", question)
    print("A:", result['result'])
    print()

ask_question("What are the main threats to coral reefs?")
ask_question("How can coral reefs be restored?")
ask_question("What is the economic value of coral reefs?")

Q: What are the main threats to coral reefs?
A: The main threats to coral reefs mentioned in the context are:

1. Climate change
2. Overfishing and destructive fishing practices (dynamite fishing, cyanide fishing)
3. Coastal development and habitat destruction
4. Ocean acidification weakening coral skeletons
5. Invasive species such as the crown-of-thorns starfish
6. Plastic pollution and marine debris

Note that the context does not provide an exhaustive list of all threats to coral reefs, but rather highlights some of the most significant ones.

Q: How can coral reefs be restored?
A: According to the context, coral reefs can be restored through:

1. Coral Gardening: Transplanting fragments of healthy coral from underwater nurseries onto degraded reefs.
2. Coral Gene Banking: Preserving genetic diversity by maintaining coral fragments in controlled facilities.

Additionally, other methods mentioned for protecting coral reefs include:

1. Marine Protected Areas (MPAs)
2. Microfragmenta